# ============================================================
# TRIAGEM DAS IMAGENS
#
# Este notebook será utilizado para inspecionar todas as
# imagens do projeto, identificando aquelas que serão
# utilizadas na construção do dataset.
#
# Para cada imagem serão registradas informações como:
#
# • presença de banco de areia;
# • qualidade visual;
# • utilização ou descarte.
# ============================================================

# Bibliotecas

In [ ]:
import os
import glob
from scipy import ndimage
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
from IPython.display import clear_output

rio = gpd.read_file(
    "../data/raw/shapefiles/rio_tapajos/ana_rio_tapajos.shp"
)

frames = gpd.read_file(
    "../data/raw/geojson/frames_1280_overlay_50pc.geojson"
)

# Ler todas as imagens

In [ ]:
imagens = sorted(
    glob.glob("../data/raw/images/*.tif"),
    key=lambda caminho: int(
        os.path.splitext(
            os.path.basename(caminho)
        )[0]
    )
)

print(f"{len(imagens)} imagens encontradas.")

# Função para normalizar

In [ ]:
def normalizar(img):
    img = img.astype(np.float32)

    minimo = np.nanpercentile(img, 2)
    maximo = np.nanpercentile(img, 98)

    # Evita divisão por zero em imagens sem variação
    if maximo <= minimo:
        return np.zeros_like(img, dtype=np.float32)

    img = np.clip(img, minimo, maximo)

    return (img - minimo) / (maximo - minimo)

In [ ]:
def normalizar_conjunta(canais):
    """
    Normaliza todos os canais juntos.

    Essa abordagem preserva melhor a relação entre as bandas,
    sendo útil para produzir uma composição próxima das cores naturais.
    """

    imagem = np.dstack(canais).astype(np.float32)

    minimo = np.nanpercentile(imagem, 2)
    maximo = np.nanpercentile(imagem, 98)

    if maximo <= minimo:
        return np.zeros_like(imagem, dtype=np.float32)

    imagem = np.clip(imagem, minimo, maximo)

    return (imagem - minimo) / (maximo - minimo)


def calcular_indice(banda_a, banda_b):
    """
    Calcula um índice normalizado evitando divisão por zero.
    """

    banda_a = banda_a.astype(np.float32)
    banda_b = banda_b.astype(np.float32)

    denominador = banda_a + banda_b

    return np.divide(
        banda_a - banda_b,
        denominador,
        out=np.zeros_like(banda_a, dtype=np.float32),
        where=denominador != 0
    )

# Função para abrir imagem

In [ ]:
def abrir_imagem(caminho):

    with rasterio.open(caminho) as src:

        bandas = {
            "B02": src.read(1),
            "B03": src.read(2),
            "B04": src.read(3),
            "B08": src.read(4),
            "B11": src.read(5),
            "B12": src.read(6)
        }

    return bandas

# Visualizar imagem

In [ ]:
def mostrar_imagem(bandas, titulo=""):
    """
    Exibe diferentes composições da mesma imagem Sentinel-2
    para auxiliar na identificação dos bancos de areia.
    """

    # RGB próximo das cores naturais.
    # A normalização é feita sobre os três canais juntos.
    rgb_natural = normalizar_conjunta([
        bandas["B04"],
        bandas["B03"],
        bandas["B02"]
    ])

    # RGB com maior contraste.
    # Cada banda é normalizada separadamente.
    rgb_contrastado = np.dstack([
        normalizar(bandas["B04"]),
        normalizar(bandas["B03"]),
        normalizar(bandas["B02"])
    ])

    # Falsa cor com infravermelho próximo.
    # A vegetação normalmente aparece em tons de vermelho.
    falsa_cor_nir = np.dstack([
        normalizar(bandas["B08"]),
        normalizar(bandas["B04"]),
        normalizar(bandas["B03"])
    ])

    # Composição com infravermelho de ondas curtas.
    # Ajuda a diferenciar água, umidade, vegetação e solo exposto.
    composicao_swir = np.dstack([
        normalizar(bandas["B11"]),
        normalizar(bandas["B08"]),
        normalizar(bandas["B04"])
    ])

    # NDWI: água tende a apresentar valores mais altos.
    ndwi = calcular_indice(
        bandas["B03"],
        bandas["B08"]
    )

    # MNDWI: utiliza SWIR e pode separar melhor água e solo exposto.
    mndwi = calcular_indice(
        bandas["B03"],
        bandas["B11"]
    )

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(18, 12)
    )

    axes[0, 0].imshow(rgb_natural)
    axes[0, 0].set_title("RGB natural")

    axes[0, 1].imshow(rgb_contrastado)
    axes[0, 1].set_title("RGB contrastado")

    axes[0, 2].imshow(falsa_cor_nir)
    axes[0, 2].set_title("Falsa cor — NIR")

    axes[1, 0].imshow(composicao_swir)
    axes[1, 0].set_title("Composição SWIR")

    imagem_ndwi = axes[1, 1].imshow(
        ndwi,
        cmap="BrBG",
        vmin=-1,
        vmax=1
    )
    axes[1, 1].set_title("NDWI")
    fig.colorbar(
        imagem_ndwi,
        ax=axes[1, 1],
        fraction=0.046,
        pad=0.04
    )

    imagem_mndwi = axes[1, 2].imshow(
        mndwi,
        cmap="BrBG",
        vmin=-1,
        vmax=1
    )
    axes[1, 2].set_title("MNDWI")
    fig.colorbar(
        imagem_mndwi,
        ax=axes[1, 2],
        fraction=0.046,
        pad=0.04
    )

    for eixo in axes.ravel():
        eixo.axis("off")

    if titulo:
        fig.suptitle(
            titulo,
            fontsize=18,
            y=0.98
        )

    plt.tight_layout(
        rect=[0, 0, 1, 0.96]
    )

    plt.show()

# Pré-triagem automática

In [ ]:
def analisar_imagem_automaticamente(caminho):
    """
    Analisa uma imagem e atribui uma classe preliminar.

    A classificação automática não substitui a revisão manual.
    Ela serve apenas para ordenar e reduzir as imagens que
    precisam ser inspecionadas primeiro.
    """

    bandas = abrir_imagem(caminho)

    b02 = bandas["B02"].astype(np.float32)
    b03 = bandas["B03"].astype(np.float32)
    b04 = bandas["B04"].astype(np.float32)
    b08 = bandas["B08"].astype(np.float32)
    b11 = bandas["B11"].astype(np.float32)
    b12 = bandas["B12"].astype(np.float32)

    valido = (
        np.isfinite(b02)
        & np.isfinite(b03)
        & np.isfinite(b04)
        & np.isfinite(b08)
        & np.isfinite(b11)
        & np.isfinite(b12)
    )

    percentual_valido = valido.mean() * 100

    if percentual_valido < 80:
        return {
            "percentual_valido": percentual_valido,
            "percentual_candidato": 0.0,
            "maior_regiao_pixels": 0,
            "numero_regioes": 0,
            "pontuacao": 0,
            "classe_automatica": "ruim"
        }

    ndwi = calcular_indice(b03, b08)
    mndwi = calcular_indice(b03, b11)
    ndvi = calcular_indice(b08, b04)

    # Bare Soil Index: ajuda a diferenciar solo/areia de vegetação
    numerador_bsi = (b11 + b04) - (b08 + b02)
    denominador_bsi = (b11 + b04) + (b08 + b02)

    bsi = np.divide(
        numerador_bsi,
        denominador_bsi,
        out=np.zeros_like(b04, dtype=np.float32),
        where=np.abs(denominador_bsi) > 1e-6
    )

    # Percentis calculados apenas nos pixels válidos
    limite_b04 = np.nanpercentile(b04[valido], 65)
    limite_b11 = np.nanpercentile(b11[valido], 65)

    # Pixels com comportamento compatível com superfície exposta
    candidato = (
        valido

        # Não deve apresentar comportamento de água
        & (ndwi < 0)
        & (mndwi < 0)

        # Exclusão mais forte da vegetação
        & (ndvi < 0.20)

        # Solo/areia exposta tende a apresentar BSI positivo
        & (bsi > 0)

        # Regiões relativamente claras
        & (b04 >= limite_b04)
        & (b11 >= limite_b11)
    )

    # Remove pixels isolados e pequenos ruídos
    candidato = ndimage.binary_opening(
        candidato,
        structure=np.ones((2, 2), dtype=bool)
    )

    candidato = ndimage.binary_closing(
        candidato,
        structure=np.ones((2, 2), dtype=bool)
    )


    # Recalcula as regiões depois da remoção das bordas
    rotulos, numero_regioes = ndimage.label(candidato)

    if numero_regioes > 0:
        tamanhos = ndimage.sum(
            candidato,
            rotulos,
            index=np.arange(1, numero_regioes + 1)
        )

        maior_regiao = int(np.max(tamanhos))

    else:
        maior_regiao = 0

    percentual_candidato = (
        candidato.sum() / valido.sum() * 100
    )

    pontuacao = 0

    if percentual_candidato >= 0.10:
        pontuacao += 1

    if percentual_candidato >= 0.50:
        pontuacao += 1

    if maior_regiao >= 5:
        pontuacao += 1

    if maior_regiao >= 15:
        pontuacao += 1

    if numero_regioes >= 1:
        pontuacao += 1

    if pontuacao >= 4:
        classe = "provavel_banco"

    elif pontuacao >= 2:
        classe = "revisar"

    else:
        classe = "provavel_sem_banco"

    return {
        "percentual_valido": percentual_valido,
        "percentual_candidato": percentual_candidato,
        "maior_regiao_pixels": maior_regiao,
        "numero_regioes": int(numero_regioes),
        "pontuacao": pontuacao,
        "classe_automatica": classe
    }

# Teste


In [ ]:
for id_imagem in ["2184", "2185", "2186"]:
    caminho = (
        f"../data/raw/images/{id_imagem}.tif"
    )

    resultado = analisar_imagem_automaticamente(
        caminho
    )

    print(
        id_imagem,
        resultado["classe_automatica"],
        resultado["pontuacao"],
        resultado["maior_regiao_pixels"],
        resultado["percentual_candidato"]
    )

In [ ]:
caminho_triagem_automatica = (
    "../data/processed/triagem_automatica.csv"
)

os.makedirs(
    os.path.dirname(caminho_triagem_automatica),
    exist_ok=True
)

if os.path.exists(caminho_triagem_automatica):
    triagem_automatica = pd.read_csv(
        caminho_triagem_automatica
    )

    print(
        "Pré-triagem automática já existente: "
        f"{len(triagem_automatica)} imagens."
    )

else:
    resultados_automaticos = []

    for indice, caminho_imagem in enumerate(
        imagens,
        start=1
    ):
        id_imagem = os.path.splitext(
            os.path.basename(caminho_imagem)
        )[0]

        resultado = analisar_imagem_automaticamente(
            caminho_imagem
        )

        resultado["id"] = id_imagem
        resultado["caminho"] = caminho_imagem

        resultados_automaticos.append(resultado)

        if indice % 100 == 0:
            print(
                f"{indice}/{len(imagens)} "
                "imagens analisadas"
            )

    triagem_automatica = pd.DataFrame(
        resultados_automaticos
    )

    triagem_automatica.to_csv(
        caminho_triagem_automatica,
        index=False
    )

    print("Pré-triagem automática concluída.")

In [ ]:
print("Resultado da pré-triagem automática:")
print("-" * 45)

display(
    triagem_automatica[
        "classe_automatica"
    ].value_counts()
)

display(
    triagem_automatica.sort_values(
        [
            "pontuacao",
            "maior_regiao_pixels",
            "percentual_candidato"
        ],
        ascending=False
    ).head(20)
)

In [ ]:
classes_para_revisar = [
    "provavel_banco",
    "revisar"
]

candidatas = triagem_automatica[
    triagem_automatica["classe_automatica"].isin(
        classes_para_revisar
    )
].copy()

candidatas = candidatas.sort_values(
    [
        "pontuacao",
        "maior_regiao_pixels",
        "percentual_candidato"
    ],
    ascending=False
)

# Remove as imagens que você já avaliou
candidatas_pendentes = candidatas[
    ~candidatas["id"].astype(str).isin(ids_avaliados)
].copy()

# Pega somente as próximas 200 ainda não avaliadas
candidatas_top_200 = candidatas_pendentes.head(200).copy()

imagens_para_revisar = candidatas_top_200[
    "caminho"
].tolist()

print(f"Imagens originais: {len(imagens)}")
print(f"Candidatas automáticas: {len(candidatas)}")
print(f"Já avaliadas: {len(ids_avaliados)}")
print(f"Candidatas pendentes: {len(candidatas_pendentes)}")
print(f"Selecionadas agora: {len(imagens_para_revisar)}")

In [ ]:
# Seleciona uma amostra aleatória das imagens classificadas
# automaticamente como "provável sem banco"

negativas_teste = triagem_automatica[
    triagem_automatica["classe_automatica"]
    == "provavel_sem_banco"
].sample(
    n=min(20, (
        triagem_automatica["classe_automatica"]
        == "provavel_sem_banco"
    ).sum()),
    random_state=42
)

imagens_negativas_teste = negativas_teste[
    "caminho"
].tolist()

print(
    f"{len(imagens_negativas_teste)} imagens negativas "
    "selecionadas para validação."
)

In [ ]:
for indice, caminho_imagem in enumerate(
    imagens_para_revisar,
    start=1
):

    id_imagem = os.path.splitext(
        os.path.basename(caminho_imagem)
    )[0]


    # Remove a figura e os textos da imagem anterior
    clear_output(wait=True)
    plt.close("all")

    bandas = abrir_imagem(caminho_imagem)

    mostrar_imagem(
        bandas,
        titulo=(
            f"Imagem {id_imagem} | "
            f"{indice} de {len(imagens_para_revisar)}"
        )
    )

    resposta = input(
        "A imagem possui banco de areia? "
        "[s = sim / n = não / q = encerrar]: "
    ).strip().lower()

    plt.close("all")

    if resposta == "q":
        break

    if resposta == "s":
        print(
            f"ATENÇÃO: a imagem {id_imagem} foi classificada "
            "automaticamente como negativa, mas parece conter banco."
        )
    else:
        print(f"Imagem {id_imagem}: negativo confirmado.")

In [ ]:
# Caminho do arquivo que armazenará as decisões
caminho_triagem = "../data/processed/triagem.csv"

# Garante que a pasta exista
os.makedirs(
    os.path.dirname(caminho_triagem),
    exist_ok=True
)

# Se já existir uma triagem, continua de onde parou
if os.path.exists(caminho_triagem):
    triagem = pd.read_csv(caminho_triagem)

    # Converte os IDs já avaliados para conjunto,
    # facilitando a busca
    ids_avaliados = set(
        triagem["id"].astype(str)
    )

    print(
        f"Triagem existente carregada: "
        f"{len(triagem)} imagens avaliadas."
    )

else:
    triagem = pd.DataFrame(
        columns=[
            "id",
            "possui_banco",
            "qualidade",
            "usar_dataset",
            "observacao"
        ]
    )

    ids_avaliados = set()

    print("Nova triagem criada.")

In [ ]:
def registrar_avaliacao(
    id_imagem,
    possui_banco,
    qualidade,
    usar_dataset,
    observacao=""
):
    """
    Registra ou atualiza a avaliação de uma imagem
    e salva imediatamente no arquivo CSV.
    """

    global triagem
    global ids_avaliados

    id_imagem = str(id_imagem)

    nova_linha = {
        "id": id_imagem,
        "possui_banco": possui_banco,
        "qualidade": qualidade,
        "usar_dataset": usar_dataset,
        "observacao": observacao
    }

    # Atualiza a linha se o ID já existir
    if id_imagem in triagem["id"].astype(str).values:
        indice = (
            triagem["id"]
            .astype(str)
            .eq(id_imagem)
        )

        for coluna, valor in nova_linha.items():
            triagem.loc[indice, coluna] = valor

    # Caso contrário, adiciona uma nova linha
    else:
        triagem.loc[len(triagem)] = nova_linha

    ids_avaliados.add(id_imagem)

    # Salva após cada imagem para não perder o trabalho
    triagem.to_csv(
        caminho_triagem,
        index=False
    )

In [ ]:
for indice, caminho_imagem in enumerate(
    imagens_para_revisar,
    start=1
):

    id_imagem = os.path.splitext(
        os.path.basename(caminho_imagem)
    )[0]

    # Ignora imagens já avaliadas
    if id_imagem in ids_avaliados:
        continue

    bandas = abrir_imagem(caminho_imagem)

    mostrar_imagem(
        bandas,
        titulo=(
            f"Imagem {id_imagem} | "
            f"{indice} de {len(imagens_para_revisar)}"
        )
    )

    print("\nClassifique a imagem:")
    print("1 - Possui banco de areia e qualidade boa")
    print("2 - Não possui banco de areia e qualidade boa")
    print("3 - Imagem ruim ou inadequada")
    print("4 - Possui banco, mas precisa de revisão")
    print("0 - Encerrar a triagem")

    while True:
        escolha = input("Escolha: ").strip()

        if escolha in {"0", "1", "2", "3", "4"}:
            break

        print("Opção inválida. Digite 0, 1, 2, 3 ou 4.")

    if escolha == "0":
        print("Triagem encerrada.")
        break

    if escolha == "1":
        registrar_avaliacao(
            id_imagem=id_imagem,
            possui_banco=True,
            qualidade="boa",
            usar_dataset=True
        )

    elif escolha == "2":
        registrar_avaliacao(
            id_imagem=id_imagem,
            possui_banco=False,
            qualidade="boa",
            usar_dataset=True
        )

    elif escolha == "3":
        observacao = input(
            "Motivo do descarte, opcional: "
        ).strip()

        registrar_avaliacao(
            id_imagem=id_imagem,
            possui_banco=False,
            qualidade="ruim",
            usar_dataset=False,
            observacao=observacao
        )

    elif escolha == "4":
        observacao = input(
            "O que precisa ser revisado? "
        ).strip()

        registrar_avaliacao(
            id_imagem=id_imagem,
            possui_banco=True,
            qualidade="revisar",
            usar_dataset=False,
            observacao=observacao
        )

    # Fecha a figura antes de mostrar a próxima
    plt.close("all")

    print(
        f"Imagem {id_imagem} registrada. "
        f"Total avaliado: {len(triagem)}\n"
    )

In [ ]:
print("Resumo da triagem")
print("-" * 40)

print(f"Imagens avaliadas: {len(triagem)}")

print("\nPresença de banco:")
display(
    triagem["possui_banco"].value_counts(
        dropna=False
    )
)

print("\nQualidade:")
display(
    triagem["qualidade"].value_counts(
        dropna=False
    )
)

print("\nUso no dataset:")
display(
    triagem["usar_dataset"].value_counts(
        dropna=False
    )
)

In [ ]:
triagem.tail(10)

In [ ]:
# Imagens positivas, com banco de areia e boa qualidade
positivas = triagem[
    (triagem["possui_banco"] == True)
    & (triagem["qualidade"] == "boa")
    & (triagem["usar_dataset"] == True)
].copy()

# Imagens negativas, sem banco e com boa qualidade
negativas = triagem[
    (triagem["possui_banco"] == False)
    & (triagem["qualidade"] == "boa")
    & (triagem["usar_dataset"] == True)
].copy()

print(f"Positivas disponíveis: {len(positivas)}")
print(f"Negativas disponíveis: {len(negativas)}")

In [ ]:
quantidade_negativas = min(50, len(negativas))

negativas_selecionadas = negativas.sample(
    n=quantidade_negativas,
    random_state=42
)

dataset_selecionado = pd.concat(
    [
        positivas,
        negativas_selecionadas
    ],
    ignore_index=True
)

dataset_selecionado = dataset_selecionado.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(f"Total selecionado: {len(dataset_selecionado)}")

print(
    dataset_selecionado["possui_banco"]
    .value_counts()
)

In [ ]:
caminho_selecao = (
    "../data/processed/dataset_selecionado.csv"
)

dataset_selecionado.to_csv(
    caminho_selecao,
    index=False
)

print(f"Seleção salva em: {caminho_selecao}")